# Limpieza de datos – BAI Dataset

Este notebook ejecuta la ingesta de datos, validaciones, y procesamiento para el BAI dataset.

Pasos:
- Cargar data cruda
- Revisar estructura
- Parsear respuestas
- Expandir los items del cuestionario BAI a columnas
- Validar datased
- Almacenar dataset limpio

### Setup del proyecto

In [ ]:
# Setup project root
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Asegura que src esté en sys.path
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

### Imports

In [ ]:
import importlib
import pandas as pd

from src.config import BAI_DATA_PATH, DATA_PROCESSED
from src.data_loader import load_excel
import src.preprocessing as preprocessing

importlib.reload(preprocessing)

parse_answers_column = preprocessing.parse_answers_column
expand_bai_answers = preprocessing.expand_bai_answers
prepare_bai_dataset = preprocessing.prepare_bai_dataset
validate_bai_dataset = preprocessing.validate_bai_dataset

### Cargar data cruda

In [ ]:
df_raw = load_excel(BAI_DATA_PATH)
print("Shape original:", df_raw.shape)
print("Columnas:", df_raw.columns.tolist())
df_raw.head()

### Revisar estructura de `answers`

In [ ]:
print("Tipo de dato en answers antes del parseo:", type(df_raw.loc[0, "answers"]).__name__)
print("Ejemplo de answers crudo:", df_raw.loc[0, "answers"])

### Parsear y expandir respuestas BAI

In [ ]:
df_parsed = parse_answers_column(df_raw, column="answers")

print("Tipo de dato en answers despues del parseo:", type(df_parsed.loc[0, "answers"]).__name__)
print("Longitud del arreglo answers en la primera fila:", len(df_parsed.loc[0, "answers"]))
print("Ejemplo parseado:", df_parsed.loc[0, "answers"])

In [ ]:
df_clean = expand_bai_answers(df_parsed, column="answers")

bai_columns = [f"BAI_{i}" for i in range(1, 22)]
print("Columnas BAI creadas:", bai_columns)
print(df_clean[["id", "answers", *bai_columns]].head())

### Preparar dataset final

In [ ]:
df_final = prepare_bai_dataset(df_clean)

print("Columnas eliminadas del dataset final: ['answers', 'email', 'name']")
print("Shape final:", df_final.shape)
print(df_final.head())

### Validar y guardar dataset procesado

In [ ]:
validate_bai_dataset(df_final)

DATA_PROCESSED.parent.mkdir(parents=True, exist_ok=True)
df_final.to_excel(DATA_PROCESSED, index=False)

print("Validacion completada correctamente.")
print("Archivo guardado en:", DATA_PROCESSED)